In [51]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [52]:
# -----------------------------------------------------------------
# SETUP (given)
# -----------------------------------------------------------------
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Class counts (train):\n{y_train.value_counts()}\n")
 

Train shape: (142, 13), Test shape: (36, 13)
Class counts (train):
target
1    57
0    47
2    38
Name: count, dtype: int64



In [53]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train,y_train)
y_pred = gnb.predict(X_test)
acc_score = accuracy_score(y_pred,y_test)
print(f"The accuracy score is {acc_score}")
f1_score_gnb = f1_score(y_test,y_pred,average='macro')
print(f"The f1 score is {f1_score_gnb}")
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

The accuracy score is 0.9722222222222222
The f1 score is 0.974320987654321
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        12
           1       1.00      0.93      0.96        14
           2       1.00      1.00      1.00        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36

[[12  0  0]
 [ 1 13  0]
 [ 0  0 10]]


In [54]:
from sklearn.neighbors import KNeighborsClassifier


pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
param_grid = {"knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15]}
gscv = GridSearchCV(pipeline,param_grid, cv=5) #What is this and how to use it?
gscv.fit(X_train,y_train)
print(f"The best parameter is {gscv.best_params_} and the best score is {gscv.best_score_}")
g = gscv.best_estimator_ 
score1 = accuracy_score(y_test,g.predict(X_test))
print(f"The accuracy score is {score1}")

The best parameter is {'knn__n_neighbors': 11} and the best score is 0.9721674876847292
The accuracy score is 1.0


In [55]:
from sklearn.tree import DecisionTreeClassifier


dtc = DecisionTreeClassifier(criterion='entropy', random_state=42)
dtc.fit(X_train,y_train)
y_pred = dtc.predict(X_test)
dtc_acc = accuracy_score(y_test,y_pred)
print(f"The accuracy is {dtc_acc}")
print(dtc.get_depth) #I think I did this wrong
print(dtc.get_n_leaves)# this too
feat = pd.Series(dtc.feature_importances_, index = X.columns)
print(feat.sort_values(ascending=False).head(5))

The accuracy is 0.9722222222222222
<bound method BaseDecisionTree.get_depth of DecisionTreeClassifier(criterion='entropy', random_state=42)>
<bound method BaseDecisionTree.get_n_leaves of DecisionTreeClassifier(criterion='entropy', random_state=42)>
flavanoids           0.403086
proline              0.288063
color_intensity      0.248159
alcalinity_of_ash    0.030182
alcohol              0.018118
dtype: float64


In [56]:
from sklearn.ensemble import RandomForestClassifier


rfc = RandomForestClassifier(n_estimators=200, random_state=42, oob_score=True)#How do we know what parameters to use? Will they be specified in the questions? Is there some way to infer?
rfc.fit(X_train,y_train)
y_pred = rfc.predict(X_test)
rfc_acc = accuracy_score(y_pred,y_test)
print(f"Accuracy score: {rfc_acc}")
print(f"OOB score: {rfc.oob_score_}")# The underscore is a different attribute
print(rfc.max_features) #sqlearn default is square root

Accuracy score: 1.0
OOB score: 0.9788732394366197
sqrt


In [ ]:
from sklearn.ensemble import AdaBoostClassifier


ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train,y_train)
y_pred = ada.predict(X_test)
ada_acc = accuracy_score(y_pred,y_test)
print(f"Accuracy: {ada_acc}")
ada_f1 = f1_score(y_pred,y_test,average='macro')
print(f"Macro f1 : {ada_f1}")
print(classification_report(y_pred,y_test))
#class 1 looks worse

Accuracy: 0.9166666666666666
Macro f1 : 0.9197780776728145
              precision    recall  f1-score   support

           0       1.00      0.86      0.92        14
           1       0.86      0.92      0.89        13
           2       0.90      1.00      0.95         9

    accuracy                           0.92        36
   macro avg       0.92      0.93      0.92        36
weighted avg       0.92      0.92      0.92        36



In [66]:
models = {
    "Naive Beyes":gnb,
    "knn":gscv,
    "Decision Tree": dtc,
    "Random Forest":rfc,
    "AdaBoost": ada
}
sorted_macro = {}

skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
for name, model in models.items():
    if(name == "knn"):
        cross_score = cross_val_score(Pipeline([
            ('scaler',StandardScaler()),
            ('knn',KNeighborsClassifier())
        ]),X,y,cv = skf,scoring = "f1_macro")
    else:
        cross_score = cross_val_score(model,X,y,cv = skf,scoring = "f1_macro")
    sorted_macro[name] = cross_score.mean()

feat = pd.Series(sorted_macro)
feat2 = feat.sort_values(ascending=False).head(5)
print(feat2)

Random Forest    0.978395
Naive Beyes      0.973204
knn              0.972109
AdaBoost         0.928614
Decision Tree    0.908947
dtype: float64


In [72]:
best_name = max(sorted_macro, key = sorted_macro.get)
print(best_name)

best_model = models[best_name]
best_model.fit(X_train,y_train)
best_model_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test,best_model_pred)
print(cm)

Random Forest
[[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]
